**Imports**

In [11]:
import csv #bilt in csv reader
from pathlib import Path# modern - file handling
from typing import Annotated, Sequence, TypedDict # used to define AgentState 

from dotenv import load_dotenv
#into .env file so that API key loads into environment so that ChatGroq finds it

#llamaindex powers theFAISS search 
import faiss # faiss-> vector daatbase
from llama_index.core import Document, Settings, VectorStoreIndex, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore
# document -> wraps CSV rows with text + metadata
# Settings -> global config for the embedder
#vectorstoreIndex-> orchestrate embedding + storage
#storage context=> tells llamaIndex "store vectors in this FAISS instance"
#HuggingfaceEmbedding-> turns text into 384-number vectors
#FAISSvectorStore-> adapter so LlamaIndex can talk to FAISS

#LangChain/ langGraph = powers the ReAct agent 
from langchain_core.messages import BaseMessage, SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq# groq ka 
from langgraph.graph import StateGraph, END # staegraph orchestrate the whole execution of langgraph
from langgraph.graph.message import add_messages# reducer- just append the next msg dont overwrite
from langgraph.prebuilt import ToolNode#node that actually executes tool calls
#SystemMessage- instruction to llm "You are my assistant"
#HumanMessage- what the user said
#AIMessage- what LLM replied
#ToolMessage- result of tool call
#BaseMessage-parent class of all the above

load_dotenv()
#.env puts groq_api_key into os.environ now when we craete ChatGroq() it auto discover it


True

**Build the FAISS retriever**

In [12]:
def build_retriever():
    """Load complaints.csv -> embed everything-> return a retriever"""

    Settings.embed_model = HuggingFaceEmbedding(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    Settings.llm = None

    CSV_PATH = Path("complaints.csv")
    documents = []
    with open(CSV_PATH, "r") as f:
        for i, row in enumerate(csv.DictReader(f)):
            documents.append(Document(
                text=row["complaint_text"],
                metadata={
                    "product_area": row["product_area"],
                    "root_cause_label": row["root_cause_label"],
                    "capa_summary": row["capa_summary"],
                    "row_id": i,
                },
            ))

    faiss_index = faiss.IndexFlatL2(384)
    vector_store = FaissVectorStore(faiss_index=faiss_index)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True,
    )
    print(f"Indexed {faiss_index.ntotal} complaints")
    return index.as_retriever(similarity_top_k=5)


retriever = build_retriever()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM is explicitly disabled. Using MockLLM.


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/71 [00:00<?, ?it/s]

Indexed 71 complaints


**Wrap the retriever as a tool**

In [13]:
@tool
def search_complaints(query: str):
    """Search the database of past manufacturing complaints

    Use this tool whenever the user asks about defects, root causes, CAPAs, suppliers or past problem in the factory

    Args: 
        query: A natural-language description of what to search so far

    Returns:
        The top 5 most similar past complaints each with product area, root cause label , and CAPA summary
    
    """
    #docstring is a prompt read by llm every time it decides whether to call the tool
    results=retriever.retrieve(query)

    output=[]
    for i, node in enumerate(results, start=1):#LLM will see result from result 1 ,2 ,3 instead of 0 isiliye start=1
        output.append(
            f"Result {i} (similarity {node.score:.3f}):\n"
            f" Complaint: {node.text}\n"
            f" Product Area: {node.metadata['product_area']}\n"
            f" Root Cause: {node.metadata['root_cause_label']}\n"
            f" CAPA: {node.metadata['capa_summary']}"
        )
        # indented key-value pair=> easy for the LLM to parse each field
        # similarity score included => lets the LLm say things like there's a very close matcgh
        # all 5 results concatenated with blank lines

        #we are sending all this info as a prompt so llm can answer thoroughly

    return "\n\n".join(output)#Always convert tool results to a well-formatted, human-readable string even if u r not bcoz LLM tool output as part of its next prompt. LLm always return them as a string

#Group all tools in a list - we'll pass this to LLm and the graph
tools=[search_complaints]# why list bcoz later we will do bind tools which expect a list

# #sanity check
# print(search_complaints.invoke("supplier defects on PCB boards"))

# ── DEBUG: what does the retriever actually return? ──
results = retriever.retrieve("supplier defects on PCB boards")

print(f"Number of results: {len(results)}\n")

# Look at the very first result in detail
first = results[0]

print("=" * 60)
print("TYPE of node:")
print(type(first))

print("\n" + "=" * 60)
print("ALL METADATA KEYS:")
print(list(first.metadata.keys()))

print("\n" + "=" * 60)
print("FULL METADATA DICT:")
print(first.metadata)

print("\n" + "=" * 60)
print("TEXT (first 200 chars):")
print(first.text[:200])


     

Number of results: 5

TYPE of node:
<class 'llama_index.core.schema.NodeWithScore'>

ALL METADATA KEYS:
['product_area', 'root_cause_label', 'capa_summary', 'row_id']

FULL METADATA DICT:
{'product_area': 'Electronics', 'root_cause_label': 'supplier_defect', 'capa_summary': 'Quarantined 500 boards. Issued deviation report. Supplier re-qualified with updated Gerber file verification step.', 'row_id': 2}

TEXT (first 200 chars):
PCB boards from Supplier B have incorrect trace routing — does not match approved Gerber files revision 3.2.


In [14]:
#agent's memory shape
#blueprint that says agent's state is a dict with one key called messages which is a list of msg objects
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]#seq{baseMessaage}=> field hold a list of msg
    #annotated[]=> when node return new msg use add_msg to merge them in
    #add_msg: function that append new msg instead of overwriting them

#LLM
model=ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0# we want deterministic tools not random answers creative answers
).bind_tools(tools)# tells llm these are the tools you can use w/o it llm can use the tools from their own knowledge

#Node 1: ask the LLm what to do next
def model_call(state: AgentState) -> AgentState:
    system_prompt=SystemMessage(content=(
        "You are a quality engineering assistant for a manufacturing company. "
        "When the user asks about complaints, defects, suppliers, root causes, "
        "or CAPAs, ALWAYS call the search_complaints tool first. "
        "After you get results, cite the product_area and root_cause_label "
        "when answering."
    ))
    response=model.invoke([system_prompt]+state["messages"])
    return {"messages": [response]}

#decision function
def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if not last_message.tool_calls:
        return "end"
    return "continue"

#

**why do we @tool as decorator**
so llms require text whereas python is coding we need to extract name(function name) description(docstring) and argument(qery) 
decorator returns a wrapped version of a function with extra behaviour added wo changing the original code

In [15]:
# Create the graph with our state blueprint
graph=StateGraph(AgentState)

#Add the 2 nodes 
graph.add_node("our_agent", model_call)# laberl name for model call is our_agent
graph.add_node("tools", ToolNode(tools=tools))#creates an instance that acts like a function when the graph calls it with state, ToolNode
#1. Read last msg-> extract tool_name and args -> find matching tool in tools list -> calls it with those arguments -> return the result wrapped as a toolMessage

#Tell the graph where to start
graph.set_entry_point("our_agent")

#decision need to be made
graph.add_conditional_edges(
    "our_agent",
    should_continue,
    {
        "continue": "tools",
        "end":END
    }
)
graph.add_edge("tools", "our_agent")

app=graph.compile()# using compile we have an executable that can be invoke and traced



           START
             │
             ▼
       ┌──────────┐
       │ our_agent│  (model_call runs here — LLM thinks)
       └─────┬────┘
             │
       should_continue?
        ┌────┴────┐
        ▼         ▼
   "continue"   "end"
        │         │
        ▼         ▼
    ┌──────┐   ┌─────┐
    │tools │   │ END │
    └──┬───┘   └─────┘
       │
       └──→ (back up to our_agent)


In [ ]:
#unlike invoke where it just returns here we have stream whicvh will give me each line
def print_stream(stream):
    """Pretty-print every step of the agent's thinking."""
    for s in stream:
        message = s["messages"][-1]#grab last msg=> print 
        if isinstance(message, tuple):
            print(message)
            #Edge case. The very first message you passed in was ("user", question) — a Python tuple, not a proper HumanMessage object. LangGraph converts it internally, but on the first yielded snapshot it might still be in tuple form. Tuples don't have a .pretty_print() method — calling it would crash. So: if it's a raw tuple, just print() it normally.
        else:
            message.pretty_print()


# Ask the agent a question
question = "Have we seen supplier issues with PCB boards?"

inputs = {"messages": [("user", question)]}
print_stream(app.stream(inputs, stream_mode="values"))


================================ Human Message =================================

Have we seen supplier issues with PCB boards?
================================== Ai Message ==================================
Tool Calls:
  search_complaints (gf94cs830)
 Call ID: gf94cs830
  Args:
    query: supplier issues with PCB boards
================================= Tool Message =================================
Name: search_complaints

Result 1 (similarity 0.821):
 Complaint: PCB boards from Supplier B have incorrect trace routing — does not match approved Gerber files revision 3.2.
 Product Area: Electronics
 Root Cause: supplier_defect
 CAPA: Quarantined 500 boards. Issued deviation report. Supplier re-qualified with updated Gerber file verification step.

Result 2 (similarity 1.001):
 Complaint: Customer reports burnt smell from power supply unit. Inspection shows discoloration on PCB near voltage regulator. Could be component rated too low (design), counterfeit component from supplier, or so